In [1]:
from pathlib import Path
import sys
parent_dir = Path.cwd().parent
sys.path.insert(0, str(parent_dir))
from config import BIGQUERY_API_KEY
from google.cloud import bigquery
from google.api_core.exceptions import GoogleAPIError

client = bigquery.Client(project="proven-reality-499800-u9")



/Users/lloydtodaro/anaconda3/envs/nascar-visibility/lib/python3.11/site-packages/google/auth/_default.py:113: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


In [9]:
# initialize everything

from pathlib import Path

SPONSORS = ["Progressive", "Cheddar's Scratch Kitchen", "Love's Travel Stops", "Busch Light", "Castrol"]

SEASONS = [
    ("2025-02-01", "2025-12-01"),
    ("2026-02-01", "2026-12-01"),
]

OUTPUT_CSV = Path("data/raw/sponsor_mentions.csv")
CHECKPOINT_FILE = Path("sponsor_mentions_checkpoint.json")
FAILED_LOG = Path("sponsor_mentions_failed.json")

MAX_GB_WARNING = 5.0  # print a warning if a single query scans more than this

QUERY = """
    SELECT
      DATE,
      DocumentIdentifier AS url,
      SourceCommonName AS source,
      V2Organizations,
      V2Tone
    FROM `gdelt-bq.gdeltv2.gkg_partitioned`
    WHERE _PARTITIONTIME >= TIMESTAMP(@start_date)
      AND _PARTITIONTIME <  TIMESTAMP(@end_date)
      AND V2Organizations LIKE @sponsor_pattern
      AND (LOWER(V2Organizations) LIKE '%nascar%' OR LOWER(AllNames) LIKE '%nascar%')
"""


In [4]:
import pandas as pd

def month_windows(start_str, end_str):
    """Break a date range into a list of (start, end) month-long window strings."""
    start = pd.Timestamp(start_str)
    end = pd.Timestamp(end_str)
    windows = []
    cur = start
    while cur < end:
        nxt = min(cur + pd.offsets.MonthBegin(1), end)
        windows.append((cur.strftime("%Y-%m-%d"), nxt.strftime("%Y-%m-%d")))
        cur = nxt
    return windows

In [5]:
import json
def load_checkpoint():
    """Load the set of (sponsor, start, end) jobs that already completed successfully."""
    if CHECKPOINT_FILE.exists():
        return set(tuple(x) for x in json.loads(CHECKPOINT_FILE.read_text()))
    return set()

 
def save_checkpoint(done):
    CHECKPOINT_FILE.write_text(json.dumps([list(x) for x in done]))
    
def log_failure(sponsor, start, end, error):
    failures = []
    if FAILED_LOG.exists():
        failures = json.loads(FAILED_LOG.read_text())
    failures.append({"sponsor": sponsor, "start": start, "end": end, "error": str(error)})
    FAILED_LOG.write_text(json.dumps(failures, indent=2))
 
 

In [6]:
def run_query(client, sponsor, start, end):
    job_config = bigquery.QueryJobConfig(
        query_parameters=[
            bigquery.ScalarQueryParameter("start_date", "STRING", start),
            bigquery.ScalarQueryParameter("end_date", "STRING", end),
            bigquery.ScalarQueryParameter("sponsor_pattern", "STRING", f"%{sponsor}%"),
        ]
    )
 
    # Dry run first so we can warn about (and could abort on) unexpectedly large scans
    dry_config = bigquery.QueryJobConfig(
        dry_run=True,
        use_query_cache=False,
        query_parameters=job_config.query_parameters,
    )
    dry_job = client.query(QUERY, job_config=dry_config)
    gb_scanned = dry_job.total_bytes_processed / 1e9
    if gb_scanned > MAX_GB_WARNING:
        print(f"    warning: this query will scan {gb_scanned:.2f} GB")
 
    df = client.query(QUERY, job_config=job_config).to_dataframe()
    df["sponsor"] = sponsor
    return df

In [8]:
#main loop
import time

done = load_checkpoint()

all_windows = []
for season_start, season_end in SEASONS:
    all_windows.extend(month_windows(season_start, season_end))
        
total_jobs = len(SPONSORS) * len(all_windows)
job_num = 0

for sponsor in SPONSORS:
    for start, end in all_windows:
        job_num += 1
        key = (sponsor, start, end)
        
        if key in done:
            print(f"[{job_num}/{total_jobs}] skip  {sponsor} {start} to {end} (already done)")
            continue
        
        print(f"[{job_num}/{total_jobs}] query {sponsor} {start} to {end}")
        try:
            df = run_query(client, sponsor, start, end)
        except GoogleAPIError as e:
            print(f"    failed: {e}")
            log_failure(sponsor, start, end, e)
            continue
        if not df.empty:
            df["date"] = pd.to_datetime(df["DATE"], format="%Y%m%d%H%M%S")
            write_header = not OUTPUT_CSV.exists()
            df.to_csv(OUTPUT_CSV, mode="a", header=write_header, index=False)
            print(f"    found {len(df)} articles, appended to {OUTPUT_CSV}")
        else:
            print("    found 0 articles")
            
        done.add(key)
        save_checkpoint(done)
        
        time.sleep(0.3)  # small pause between requests
            
print("\nDone. Results in:", OUTPUT_CSV.resolve())
if FAILED_LOG.exists():
    print("Some queries failed - see:", FAILED_LOG.resolve())
 
            


 

[1/100] skip  Progressive 2025-02-01 to 2025-03-01 (already done)
[2/100] query Progressive 2025-03-01 to 2025-04-01


/Users/lloydtodaro/anaconda3/envs/nascar-visibility/lib/python3.11/site-packages/google/cloud/bigquery/table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


    found 0 articles
[3/100] query Progressive 2025-04-01 to 2025-05-01


/Users/lloydtodaro/anaconda3/envs/nascar-visibility/lib/python3.11/site-packages/google/cloud/bigquery/table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


    found 0 articles
[4/100] query Progressive 2025-05-01 to 2025-06-01


...

/Users/lloydtodaro/anaconda3/envs/nascar-visibility/lib/python3.11/site-packages/google/cloud/bigquery/table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


    found 0 articles
[8/100] query Progressive 2025-09-01 to 2025-10-01


/Users/lloydtodaro/anaconda3/envs/nascar-visibility/lib/python3.11/site-packages/google/cloud/bigquery/table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


    found 0 articles


KeyboardInterrupt: 

In [24]:
import sys
print(sys.executable)

/Users/lloydtodaro/anaconda3/envs/nascar-visibility/bin/python


In [25]:
%pip install db-dtypes

Note: you may need to restart the kernel to use updated packages.
